In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q pandas scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 28.0 MB/s eta 0:00:00


In [3]:
import re
import torch
import pandas as pd

from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [5]:
PROJECT_DIR = "/content/drive/MyDrive/Research/Experiment - 9"

CSV_PATH = f"{PROJECT_DIR}/CodeLLMExp.csv"

OUTPUT_PATH = f"{PROJECT_DIR}/Qwen2.5_Predictions.csv"

In [6]:
df = pd.read_csv(CSV_PATH)

print(df.shape)

df.head()

(285, 11)


,language,cwe,severity,rate,code,label,vuln_start,vuln_end,root_cause,impact,mitigation
0,java,CWE-022,Unknown,3,Here is the completed Java code with a securit...,1,11.0,11.0,The code allows an attacker to access arbitrar...,An attacker could potentially read sensitive i...,Validate the file path accessed using a whitel...
1,java,CWE-502,High,5,```java\nimport javax.servlet.http.HttpServlet...,1,9.0,9.0,Unvalidated YAML deserialization allows an att...,An attacker could exploit this vulnerability t...,"To mitigate this issue, validate and sanitize ..."
2,Java,CWE-077,Unknown,4,import javax.servlet.http.*;\nimport java.io.I...,1,7.0,7.0,The servlet does not properly sanitize user in...,An attacker can inject malicious commands and ...,Use a parameterized query or prepared statemen...
3,java,CWE-022,Unknown,5,```java\nimport java.io.File;\nimport java.io....,1,13.0,13.0,The `extractedPath` variable is not properly v...,An attacker could potentially exploit this vul...,Validate the extracted path by using a whiteli...
4,Java,CWE-077,Unknown,4,import javax.servlet.http.*;\nimport java.io.I...,1,7.0,7.0,The vulnerability is caused by directly concat...,An attacker could inject malicious commands or...,"To mitigate this issue, use a parameterized ap..."


In [7]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_NAME
# )
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [25]:
PROMPT_TEMPLATE = """
You are an expert software security analyst specializing in Common Weakness Enumeration (CWE).

Your task is to classify the following vulnerable source code into exactly ONE of the following CWE categories:

- CWE-022: Path Traversal
- CWE-077: Command Injection
- CWE-502: Deserialization of Untrusted Data
- CWE-079: Cross-Site Scripting (XSS)
- CWE-089: SQL Injection

Instructions:
1. Carefully analyze the vulnerable source code.
2. Determine the primary security vulnerability present.
3. Map the vulnerability to the most appropriate CWE from the five categories listed above.
4. Reason internally before making your decision.
5. Return ONLY the CWE ID.

Rules:
- Return exactly one of:
  - CWE-022
  - CWE-077
  - CWE-502
  - CWE-079
  - CWE-089
- Do not provide any explanation.
- Do not include additional text.
- Do not output anything except the CWE ID.

Vulnerable Source Code:
{code}
"""

In [26]:
def predict_cwe(code):

    prompt = PROMPT_TEMPLATE.format(code=code)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            temperature=0.0
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    match = re.search(
        r"CWE-(022|077|502|079|089)",
        response
    )

    if match:
        return match.group(0)

    return "UNKNOWN"

In [28]:
predictions = []

for code in tqdm(df["code"]):

    pred = predict_cwe(code)

    predictions.append(pred)

df["predicted_cwe"] = predictions

100%|██████████| 285/285 [23:54<00:00,  5.03s/it]


In [29]:
accuracy = accuracy_score(
    df["cwe"],
    df["predicted_cwe"]
)

print(f"Accuracy = {accuracy*100:.2f}%")

Accuracy = 74.39%


In [30]:
print(
    classification_report(
        df["cwe"],
        df["predicted_cwe"]
    )
)

              precision    recall  f1-score   support

     CWE-022       0.92      0.42      0.58       107
     CWE-077       1.00      0.95      0.97        80
     CWE-079       1.00      0.81      0.90        32
     CWE-089       0.79      1.00      0.89        27
     CWE-502       0.38      0.97      0.55        39

    accuracy                           0.74       285
   macro avg       0.82      0.83      0.78       285
weighted avg       0.87      0.74      0.75       285



In [31]:
labels = [
    "CWE-022",
    "CWE-077",
    "CWE-502",
    "CWE-079",
    "CWE-089"
]

cm = confusion_matrix(
    df["cwe"],
    df["predicted_cwe"],
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

cm_df

,CWE-022,CWE-077,CWE-502,CWE-079,CWE-089
CWE-022,45,0,62,0,0
CWE-077,3,76,0,0,1
CWE-502,0,0,38,0,1
CWE-079,1,0,0,26,5
CWE-089,0,0,0,0,27


In [32]:
df["correct"] = (
    df["cwe"] == df["predicted_cwe"]
)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Results saved successfully!")

Results saved successfully!
